In [3]:
import sys
sys.path.append('../../../')
from tqdm import tqdm
import os
import torch
import pandas as pd
import torch.nn.functional as F
import numpy as np
from utilities import load_embedding
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from utilities import print_exams

class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def list2df(mylist, period):
    merged_rows = []
    for i in range(0, len(mylist), period):
        merged_row = []
        for j in range(period):
            merged_row += mylist[i + j]
        merged_rows.append(merged_row)

    return pd.DataFrame(merged_rows)

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask

device = torch.device('cuda:2')


In [35]:
All_data = pd.read_csv('/home/chenyh/workspace/fluProfiler/data/reverse_test/processed/All.csv')

In [ ]:
All_data['virusPassCat'].str.rep

array(['<NONE>', '<CELL>', '<EGG>', '<BOTH>', '4'], dtype=object)

In [ ]:
# All_data = pd.read_csv('/home/chenyh/workspace/fluProfiler/data/reverse_test/processed/All.csv')
# s1 = All_data['sheet'].astype(str).str.split('-', n=1, expand=True)[0].astype(int)
# test_data = All_data[s1 == 41].copy()
train_data = pd.read_csv('/home/chenyh/workspace/fluProfiler/data/reverse_test/processed/test_2024NH/train.csv')
train_data, valid_data = train_test_split(train_data, test_size=1/9, random_state=42)     

test_data = pd.read_csv('/home/chenyh/workspace/fluProfiler/data/reverse_test/processed/test_2024NH/test.csv')
valid_dataset = fluProfiler_Dataset(valid_data)
test_dataset = fluProfiler_Dataset(test_data)
valid_dataset = DataLoader(valid_dataset,batch_size=100, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=100, shuffle=False)

ValueError: invalid literal for int() with base 10: '<'

In [13]:
embedding_df = test_data
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict = load_embedding("../../../../data/reverse_test/embedding", files=sequence_names)
emb_dict = {key: value.to(device) for key, value in emb_dict.items()}

Loading tensor: 100%|██████████| 517/517 [00:19<00:00, 27.02file/s]


In [30]:
model = torch.load('./model_value_attention/2026-02-03_12-19-47.pth', weights_only=False, map_location=device)

prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_mae, test_mse, test_pearson, test_spearman, test_R2 = print_exams(reference_ls_test, prediction_ls_test)

MAE: 0.75582
MSE: 0.94501
pearson correlation: 0.68099
spearman correlation: 0.69989
R2_score: 0.43200


In [28]:
test_data['prediction'] = prediction_ls_test

In [23]:
print_exams(test_data.loc[test_data['Type'] == 'H1N1','label'], test_data.loc[test_data['Type'] == 'H1N1','prediction'])

MAE: 0.71773
MSE: 0.74590
pearson correlation: 0.38633
spearman correlation: 0.39011
R2_score: 0.10013


(0.717733730203502,
 0.745899886401261,
 PearsonRResult(statistic=0.38632666594039944, pvalue=2.5674238969320984e-68),
 SignificanceResult(statistic=0.3901140109232448, pvalue=9.677159953925924e-70),
 0.10013123099935006)

In [24]:
print_exams(test_data.loc[test_data['Type'] == 'H3N2','label'], test_data.loc[test_data['Type'] == 'H3N2','prediction'])

MAE: 0.84110
MSE: 1.27097
pearson correlation: 0.39426
spearman correlation: 0.33280
R2_score: 0.12399


(0.8410982786542446,
 1.2709731143083587,
 PearsonRResult(statistic=0.3942604784247571, pvalue=1.4373370505691607e-104),
 SignificanceResult(statistic=0.3327993832671307, pvalue=3.0842384246308974e-73),
 0.12398779712273511)

In [29]:
print_exams(test_data.loc[test_data['Type'] == 'H1N1','label'], test_data.loc[test_data['Type'] == 'H1N1','prediction'])
print_exams(test_data.loc[test_data['Type'] == 'H3N2','label'], test_data.loc[test_data['Type'] == 'H3N2','prediction'])

MAE: 0.66889
MSE: 0.64905
pearson correlation: 0.43788
spearman correlation: 0.43149
R2_score: 0.11499
MAE: 0.82929
MSE: 1.28297
pearson correlation: 0.31879
spearman correlation: 0.30731
R2_score: 0.02560


(0.8292870241721637,
 1.2829705807022491,
 PearsonRResult(statistic=0.3187940528439347, pvalue=4.3157424851022715e-38),
 SignificanceResult(statistic=0.3073080127301108, pvalue=2.1942596763912393e-35),
 0.025596905301101147)

In [ ]:
test_data['prediction'] = prediction_ls_test
test_data.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/Figure/Fig2/fluProfiler_titer.csv', index=False)

In [1]:
import sys
sys.path.append('../../../')
from fluProfiler_models import fluProfiler, fluProfiler_Config, fluProfiler
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
import json
import torch
import pandas as pd
import torch.nn.functional as F
import numpy as np
from utilities import load_embedding, EarlyStopping
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from datetime import datetime
import pickle
from utilities import print_exams

class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4').replace('<NONE>', '5')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask

 
## read complete information
data_path = '../../../../data/reverse_test/'
season_path = 'processed/test_2024NH/'
train_data = pd.read_csv(data_path + season_path + 'train.csv')
test_data = pd.read_csv(data_path + season_path + 'test.csv')
train_data, valid_data = train_test_split(train_data, test_size=1/9, random_state=42)     

In [4]:
print(train_data.shape)
print(test_data.shape)

(63924, 19)
(2774, 19)
